# Семинар 4. Запросы к одной таблице

На этом занятии вы будете писать запросы вида `SELECT … FROM … WHERE … ORDER BY … LIMIT …`
к демобазе авиаперевозок: фильтровать строки, вычислять новые столбцы, сортировать,
искать по шаблону и собирать текст.

**Как устроен ноутбук.** Каждая задача — описание и ячейка, которая начинается с `%%sql`.
Всё, что написано в такой ячейке ниже первой строки, отправляется в PostgreSQL,
а результат показывается таблицей.

**База.** PostgreSQL 17 в Docker, демобаза авиаперевозок Postgres Pro (версия 2016 года).
Как поднять базу, открыть ноутбук и подключиться через DBeaver, написано в файле `README.md`
в корне репозитория.

Все таблицы лежат в схеме `bookings`, в запросах пишем имя вместе со схемой: `bookings.flights`.

### Таблицы демобазы, которые понадобятся

| Таблица | Одна строка — это | Нужные столбцы |
|---|---|---|
| `bookings.airports` | аэропорт | `airport_code` — код из трёх букв, `airport_name`, `city` |
| `bookings.aircrafts` | модель самолёта | `aircraft_code`, `model`, `range` — дальность, км |
| `bookings.flights` | рейс в конкретный день | `flight_id`, `flight_no`, `departure_airport`, `arrival_airport`, `aircraft_code`, `status`, `scheduled_departure`, `scheduled_arrival` |
| `bookings.ticket_flights` | перелёт по билету на конкретном рейсе | `ticket_no`, `flight_id`, `fare_conditions` — класс (`Economy`, `Comfort`, `Business`), `amount` — цена |
| `bookings.tickets` | билет пассажира | `ticket_no`, `passenger_name`, `contact_data` — контакты (телефон и email) |

Статусы рейса: `Scheduled` — продажа открыта, `On Time`, `Delayed`, `Departed`, `Arrived`, `Cancelled`.
Данные сформированы на момент 13.10.2016 17:00 — у рейсов после этой даты статус `Scheduled`.

**Схема демобазы.** Таблицы, их ключи и связи; подробное описание столбцов — в [`db/schema.md`](../db/schema.md).

![Схема демобазы авиаперевозок](../db/schema.svg)

In [1]:
import sys; sys.path.insert(0, "../jupyter")   # магия %%sql лежит в папке jupyter репозитория
%load_ext sql_magic

### Проверка подключения
Если ячейка ниже показала пять аэропортов — всё готово к работе.

In [2]:
%%sql
SELECT * FROM bookings.airports LIMIT 5;

,airport_code,airport_name,city,longitude,latitude,timezone
0,MJZ,Мирный,Мирный,114.038928,62.534689,Asia/Yakutsk
1,NBC,Бегишево,Нижнекамск,52.06,55.34,Europe/Moscow
2,NOZ,Спиченково,Новокузнецк,86.8772,53.8114,Asia/Novokuznetsk
3,NAL,Нальчик,Нальчик,43.6366,43.5129,Europe/Moscow
4,OGZ,Беслан,Владикавказ,44.6066,43.2051,Europe/Moscow


Строк: 5


# Задачи: распродажа билетов на 20 октября 2016

Авиакомпания готовит распродажу билетов на рейсы 20 октября 2016 года из Москвы и Санкт-Петербурга.
Задачи идут цепочкой: результат одной нужен в следующей.

**Памятка**

```sql
SELECT столбец, выражение AS имя        -- что вывести
FROM bookings.таблица                     -- откуда
WHERE условие AND (условие OR условие)  -- какие строки оставить
ORDER BY столбец DESC                   -- как отсортировать
LIMIT 10;                               -- сколько строк вывести
```

| Что | Как пишется | Пример |
|---|---|---|
| Значение из списка | `IN (…)`, `NOT IN (…)` | `city IN ('Москва', 'Казань')` |
| Уникальные строки | `SELECT DISTINCT` | `SELECT DISTINCT status FROM bookings.flights` |
| Округление вниз | `floor(x)` | `floor(99.9)` → `99` |
| Шаблон | `LIKE`, `ILIKE`, `~` | `model LIKE 'Boeing%'`, `email ~ '\d{4}@'` |
| Строки | `\|\|`, `initcap`, `split_part` | `split_part('IVAN PETROV', ' ', 1)` → `IVAN` |
| Дата из момента | `значение::date` | `scheduled_departure::date = '2016-10-20'` |

### Задача 1. Коды аэропортов и самолётов для акции

В таблице рейсов аэропорты и самолёты указаны только кодами. Чтобы отобрать рейсы для акции,
сначала найдите нужные коды.

1. Аэропорты Москвы и Санкт-Петербурга. Выведите **airport_code**, **airport_name**, **city**.
2. Самолёты Boeing 777-300, Sukhoi SuperJet-100 и Bombardier CRJ-200. Выведите **aircraft_code**, **model**.

In [3]:
%%sql
SELECT airport_code, airport_name, city
FROM   bookings.airports
WHERE  city IN ('Москва', 'Санкт-Петербург');

,airport_code,airport_name,city
0,LED,Пулково,Санкт-Петербург
1,SVO,Шереметьево,Москва
2,VKO,Внуково,Москва
3,DME,Домодедово,Москва


Строк: 4


In [4]:
%%sql
SELECT aircraft_code, model
FROM   bookings.aircrafts
WHERE  model IN ('Boeing 777-300', 'Sukhoi SuperJet-100', 'Bombardier CRJ-200');

,aircraft_code,model
0,773,Boeing 777-300
1,SU9,Sukhoi SuperJet-100
2,CR2,Bombardier CRJ-200


Строк: 3


**Разбор.** Москва — `SVO`, `VKO`, `DME`; Санкт-Петербург — `LED`. Самолёты — `773`, `SU9`, `CR2`.

`IN (…)` короче, чем цепочка `city = 'Москва' OR city = 'Санкт-Петербург'`.
Строки сравниваются точно: `'москва'` или `'Москва '` с пробелом ничего не найдут.

### Задача 2. Акционные рейсы

Акция действует на рейсы **20 октября 2016 года**, продажа на которые ещё открыта (статус `Scheduled`):

- **из Москвы** — все рейсы на Boeing 777-300, кроме вылетов из Внуково;
- **из Санкт-Петербурга** — рейсы на региональных самолётах Superjet и CRJ-200:
  на Superjet — если полёт по расписанию короче 1 ч 15 мин, на CRJ-200 — если короче 2 ч.

Коды аэропортов и самолётов возьмите из задачи 1. Длительность полёта —
`scheduled_arrival - scheduled_departure`, её можно сравнивать с `interval '1 hour 15 minutes'`.

Выведите **flight_id**, **flight_no**, **departure_airport**, **arrival_airport**, **aircraft_code**,
**duration** (длительность), отсортируйте по **flight_id**.

In [5]:
%%sql
SELECT flight_id, flight_no, departure_airport, arrival_airport, aircraft_code,
       scheduled_arrival - scheduled_departure AS duration
FROM   bookings.flights
WHERE  scheduled_departure::date = '2016-10-20'
  AND  status = 'Scheduled'
  AND  (    (departure_airport IN ('SVO', 'DME', 'VKO')
             AND departure_airport <> 'VKO'
             AND aircraft_code = '773')
        OR  (departure_airport = 'LED'
             AND aircraft_code IN ('SU9', 'CR2')
             AND (   (aircraft_code = 'SU9' AND scheduled_arrival - scheduled_departure < interval '1 hour 15 minutes')
                  OR (aircraft_code <> 'SU9' AND scheduled_arrival - scheduled_departure < interval '2 hours'))))
ORDER BY flight_id;

,flight_id,flight_no,departure_airport,arrival_airport,aircraft_code,duration
0,270,PG0222,DME,OVB,773,03:25:00
1,5285,PG0277,SVO,OVB,773,03:25:00
2,5320,PG0225,SVO,SVX,773,01:45:00
3,7728,PG0224,SVO,AER,773,01:45:00
4,8725,PG0132,LED,VOZ,CR2,01:30:00
5,9082,PG0571,LED,GOJ,SU9,01:10:00


Строк: 6


**Разбор.** Ответ: 6 рейсов — `270`, `5285`, `5320`, `7728` из Москвы и `8725`, `9082` из Петербурга.

Главная ловушка — скобки. `AND` выполняется раньше `OR`, поэтому без внешних скобок вокруг
«Москва OR Петербург» условия на дату и статус прилипнут только к московской части,
а петербургская возьмёт рейсы за все дни и с любым статусом: запрос вернёт 126 строк.

`scheduled_departure::date` — дата вылета без времени, в часовом поясе сессии (Москва).
Условие «кроме Внуково» можно записать и короче: `departure_airport IN ('SVO', 'DME')`.

### Задача 3. Цены со скидкой 5% и 10%

Для акционных рейсов из задачи 2 решаем, какую скидку дать: 5% или 10%.
Посчитайте цены билетов **эконом-класса** (`fare_conditions = 'Economy'`) с каждой скидкой.
Округлите вниз до целого.

Цены лежат в `bookings.ticket_flights`: одна строка — один проданный перелёт, поэтому одна и та же
цена повторяется много раз. Выведите каждую цену один раз.

Выведите **flight_id**, **amount**, **price_5**, **price_10**, отсортируйте по **flight_id** и **amount**.

In [6]:
%%sql
SELECT DISTINCT flight_id, amount,
       floor(amount * 0.95) AS price_5,
       floor(amount * 0.9)  AS price_10
FROM   bookings.ticket_flights
WHERE  flight_id IN (270, 5285, 5320, 7728, 8725, 9082)
  AND  fare_conditions = 'Economy'
ORDER BY flight_id, amount;

,flight_id,amount,price_5,price_10
0,270,27900.00,26505,25110
1,270,30700.00,29165,27630
2,5285,28000.00,26600,25200
3,5285,30800.00,29260,27720
4,5320,14400.00,13680,12960
5,5320,15800.00,15010,14220
6,7728,14000.00,13300,12600
7,7728,15400.00,14630,13860
8,8725,10500.00,9975,9450
9,8725,11500.00,10925,10350


Строк: 12


**Разбор.** Ответ: 12 строк — у каждого рейса две цены эконома.

Без `DISTINCT` строк было бы больше тысячи: цена повторяется в каждом проданном билете.
`floor` округляет вниз, `round` — к ближайшему, `ceil` — вверх; для цены со скидкой берём `floor`,
чтобы не округлить в пользу авиакомпании.

### Задача 4. Минимальная цена со скидкой

Решено давать скидку 10%. Найдите самую низкую цену эконома со скидкой 10% (с округлением вниз):

1. среди акционных рейсов **из Москвы**;
2. среди акционных рейсов **из Санкт-Петербурга**.

Выведите **flight_id**, **amount**, **price_10**.

In [7]:
%%sql
SELECT DISTINCT flight_id, amount, floor(amount * 0.9) AS price_10
FROM   bookings.ticket_flights
WHERE  flight_id IN (270, 5285, 5320, 7728)
  AND  fare_conditions = 'Economy'
ORDER BY price_10
LIMIT 1;

,flight_id,amount,price_10
0,7728,14000.00,12600


Строк: 1


In [8]:
%%sql
SELECT DISTINCT flight_id, amount, floor(amount * 0.9) AS price_10
FROM   bookings.ticket_flights
WHERE  flight_id IN (8725, 9082)
  AND  fare_conditions = 'Economy'
ORDER BY price_10
LIMIT 1;

,flight_id,amount,price_10
0,9082,8900.00,8010


Строк: 1


**Разбор.** Ответ: из Москвы — 12 600 (рейс `7728`, Шереметьево — Сочи, цена без скидки 14 000);
из Петербурга — 8 010 (рейс `9082`, Пулково — Нижний Новгород, цена без скидки 8 900).

`ORDER BY … LIMIT 1` — «самый маленький»: сортируем по возрастанию и берём первую строку.
Сортировать можно по псевдониму `price_10`: `ORDER BY` выполняется после `SELECT`.
Второй запрос отличается только списком рейсов. Найти минимум сразу для обеих групп одним запросом
научимся на следующей неделе — группировкой.

### Задача 5. Пассажиры для рассылки

Рассылку отправим по почте тем пассажирам, у которых в адресе почты перед `@` стоит полная дата
рождения — **8 цифр подряд** (например, `volkova.alina_03101973@…`).

Контакты пассажира лежат в столбце `contact_data`, адрес почты достаётся так:
`contact_data->>'email'`, телефон — `contact_data->>'phone'`.

Выведите **ticket_no**, **passenger_name**, **email**, **phone** — первые 20 строк по номеру билета.

In [9]:
%%sql
SELECT ticket_no, passenger_name,
       contact_data->>'email' AS email,
       contact_data->>'phone' AS phone
FROM   bookings.tickets
WHERE  contact_data->>'email' ~ '\d{8}@'
ORDER BY ticket_no
LIMIT 20;

,ticket_no,passenger_name,email,phone
0,0005432000990,ALINA VOLKOVA,volkova.alina_03101973@postgrespro.ru,+70582584031
1,0005432000994,IRINA ANTONOVA,antonova.irina04121972@postgrespro.ru,+70844502960
2,0005432000995,VALENTINA KUZNECOVA,kuznecova.valentina10101976@postgrespro.ru,+70268080457
3,0005432001022,LEONID ORLOV,l_orlov.13061955@postgrespro.ru,+70569828738
4,0005432001023,SVETLANA POPOVA,s.popova.18041976@postgrespro.ru,+70650410562
5,0005432001026,OLEG KUZNECOV,oleg.kuznecov-10101967@postgrespro.ru,+70154552799
6,0005432001041,VLADIMIR MIRONOV,mironov-vladimir-02021977@postgrespro.ru,+70687617377
7,0005432001044,RAISA ROMANOVA,romanova.r-18081971@postgrespro.ru,+70955400547
8,0005432001052,EGOR NIKOLAEV,e_nikolaev-07021970@postgrespro.ru,+70422011135
9,0005432001054,VLADIMIR AFANASEV,afanasev-v.25051972@postgrespro.ru,+70464902131


Строк: 20


**Разбор.** Всего таких пассажиров 67 823, выводим первые 20.

`~` — поиск по регулярному выражению: `\d{8}` — восемь цифр подряд, `@` — сразу за ними.
Через `LIKE` так не написать: в его шаблонах нет «любая цифра».

Ловушка: у 167 650 пассажиров почты нет, `contact_data->>'email'` для них — `NULL`.
Условие с `NULL` даёт «неизвестно», и `WHERE` такие строки отбрасывает — отдельно исключать их не нужно.

### Задача 6. Текст рассылки

Подготовим письма пассажирам из задачи 5. Текст по шаблону:

«{Имя}, добрый день! Распродажа билетов на 20 октября: из Москвы от {X} рублей, из Петербурга — от {Y}!»

Вместо X и Y подставьте цены из задачи 4. Имя в `passenger_name` записано заглавными
вместе с фамилией: возьмите только имя и приведите его к виду «Alina».

Выведите **ticket_no**, **email**, **message_text** — первые 20 строк по номеру билета.

In [10]:
%%sql
SELECT ticket_no,
       contact_data->>'email' AS email,
       initcap(split_part(passenger_name, ' ', 1))
         || ', добрый день! Распродажа билетов на 20 октября: из Москвы от 12600 рублей, '
         || 'из Петербурга — от 8010!' AS message_text
FROM   bookings.tickets
WHERE  contact_data->>'email' ~ '\d{8}@'
ORDER BY ticket_no
LIMIT 20;

,ticket_no,email,message_text
0,0005432000990,volkova.alina_03101973@postgrespro.ru,"Alina, добрый день! Распродажа билетов на 20 октября: из Москвы от 12600 рублей, из Петербурга — от 8010!"
1,0005432000994,antonova.irina04121972@postgrespro.ru,"Irina, добрый день! Распродажа билетов на 20 октября: из Москвы от 12600 рублей, из Петербурга — от 8010!"
2,0005432000995,kuznecova.valentina10101976@postgrespro.ru,"Valentina, добрый день! Распродажа билетов на 20 октября: из Москвы от 12600 рублей, из Петербурга — от 8010!"
3,0005432001022,l_orlov.13061955@postgrespro.ru,"Leonid, добрый день! Распродажа билетов на 20 октября: из Москвы от 12600 рублей, из Петербурга — от 8010!"
4,0005432001023,s.popova.18041976@postgrespro.ru,"Svetlana, добрый день! Распродажа билетов на 20 октября: из Москвы от 12600 рублей, из Петербурга — от 8010!"
5,0005432001026,oleg.kuznecov-10101967@postgrespro.ru,"Oleg, добрый день! Распродажа билетов на 20 октября: из Москвы от 12600 рублей, из Петербурга — от 8010!"
6,0005432001041,mironov-vladimir-02021977@postgrespro.ru,"Vladimir, добрый день! Распродажа билетов на 20 октября: из Москвы от 12600 рублей, из Петербурга — от 8010!"
7,0005432001044,romanova.r-18081971@postgrespro.ru,"Raisa, добрый день! Распродажа билетов на 20 октября: из Москвы от 12600 рублей, из Петербурга — от 8010!"
8,0005432001052,e_nikolaev-07021970@postgrespro.ru,"Egor, добрый день! Распродажа билетов на 20 октября: из Москвы от 12600 рублей, из Петербурга — от 8010!"
9,0005432001054,afanasev-v.25051972@postgrespro.ru,"Vladimir, добрый день! Распродажа билетов на 20 октября: из Москвы от 12600 рублей, из Петербурга — от 8010!"


Строк: 20


**Разбор.** `split_part(строка, ' ', 1)` — первая часть строки до пробела, то есть имя;
`initcap` делает первую букву заглавной, остальные строчными: `ALINA` → `Alina`.

Числа из задачи 4 пока вписываем руками. Подставлять результат одного запроса в другой
научимся позже — подзапросами.